# Segment Anything Model (SAM) for Coastal Segmentation

# ✨ Colab Settings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alejandro-ZZ/coastal-segmentation/blob/master/SAM_workflow.ipynb)

In [ ]:
from google.colab import drive
from pathlib import Path

# Define the Google Drive paths
drive_dir = "/content/drive"
HOME = Path("/content/drive/MyDrive/UNS_Argentina/Research/Development")
drive.mount(drive_dir)

assert HOME.is_dir(), f"Home path does not exists: '{HOME}'"
print(f"HOME path: '{HOME}'")

In [ ]:
# GPU information
!nvidia-smi

Mon Sep  1 17:05:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# ✨ SAM Setup

Download model wheight and API package

In [ ]:
#@title ✔️ Model weights


# All available checkpoint links
weight_links = {
    "vit_h": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
    "vit_l": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",
    "vit_b": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
}

# Checkpoint to download
MODEL_TYPE = "vit_h"
weights_url = weight_links[MODEL_TYPE]
weights_filename = Path(weights_url).name

# Download weights
WEIGHTS_FILE = HOME / "data/models/weights" / weights_filename
WEIGHTS_FILE.parent.mkdir(parents=True, exist_ok=True)
if not WEIGHTS_FILE.exists():
    !wget -q {weights_url} -P {weights_path}
    print(f"Weights downloaded at '.../{WEIGHTS_FILE.relative_to(HOME)}'")
else:
    print(f"Weights already exist at '.../{WEIGHTS_FILE.relative_to(HOME)}'")

Weights already exist at '.../data/models/weights/sam_vit_h_4b8939.pth'


In [ ]:
#@title ✔️ API install

print("----  [ INSTALLING ] segment-anything.git  ----")
!pip install -q "git+https://github.com/facebookresearch/segment-anything.git"

print("----  [ INSTALLING ] imagecodecs  ----")
!pip install imagecodecs

  Preparing metadata (setup.py) ... done


# ✨ Project Config

In [ ]:
#@title ✔️ Imports

import cv2
import logging
import matplotlib.pyplot as plt
import numpy
import os
import pandas
import scipy
import skimage
import sys
import time
import torch
import warnings

from pathlib import Path
from scipy.spatial import cKDTree
from segment_anything import sam_model_registry, SamPredictor
from skimage import io
from typing import Any, Dict, Optional, List, Literal, Sequence, Tuple, Union


print(f"""Package versions:
     OpenCV: {cv2.__version__}
    Pytorch: {torch.__version__}
      Scipy: {scipy.__version__}
    Skimage: {skimage.__version__}
""")

Versions:
    * OpenCV:   4.12.0
    * Pytorch:  2.8.0+cu126
    * Scipy:    1.16.1
    * Skimage:  0.25.2



In [ ]:
#@title ✔️ Logging setup


def setup_logging(
        console_level: int = logging.INFO,
        debug_filename: Optional[str] = None,
        info_filename: Optional[str] = None,
        warning_filename: Optional[str] = None,
        log_format: str = "%(asctime)s - %(levelname)s - %(funcName)s - %(message)s",
        silent_names: Optional[list[str]] = None
):
    """
    Set up logging with dynamic filenames for each module.

    Parameters
    ----------
    console_level : int
        Minimum level for logs to be printed to the console.
        Default is ``logging.INFO`` (20).

    debug_filename : str, optional
        File path for DEBUG level logs. If not provided, no file handler for DEBUG level is created.

    info_filename : str, optional
        File path for INFO level logs. If not provided, no file handler for INFO level is created.

    warning_filename : str, optional
        File path for WARNING level logs. If not provided, no file handler for WARNING level is created.

    log_format : str, optional
        Format for log messages. Default is "%(asctime)s - %(levelname)s - %(funcName)s - %(message)s".

    silent_names : list[str], optional
        List of logger names that should not produce any output.
        If provided, these loggers will be set to the WARNING level.
    """
    # log_format = "%(levelname)s:%(name)s:%(message)s"
    # log_format = "%(asctime)s - %(levelname)s - %(name)s - %(filename)s:%(lineno)d - %(message)s"

    root_logger = logging.getLogger()               # Create a base logger with the root name
    root_logger.setLevel(logging.DEBUG)             # Base level for all logs
    root_logger.handlers.clear()                    # Clear any existing handlers
    formatter = logging.Formatter(log_format)       # Formatter for log messages

    # Console handler (INFO level and above, by default)
    console_handler = logging.StreamHandler(stream=sys.stdout)
    console_handler.setLevel(console_level)
    console_handler.setFormatter(formatter)
    root_logger.addHandler(console_handler)

    # File handler for DEBUG level (DEBUG and above)
    if debug_filename is not None:
        debug_file_handler = logging.FileHandler(debug_filename)
        debug_file_handler.setLevel(logging.DEBUG)
        debug_file_handler.setFormatter(formatter)
        root_logger.addHandler(debug_file_handler)

    # File handler for INFO level (INFO and above)
    if info_filename is not None:
        info_file_handler = logging.FileHandler(info_filename)
        info_file_handler.setLevel(logging.INFO)
        info_file_handler.setFormatter(formatter)
        root_logger.addHandler(info_file_handler)

    # File handler for WARNING level (WARNING and above)
    if warning_filename is not None:
        warning_file_handler = logging.FileHandler(warning_filename)
        warning_file_handler.setLevel(logging.WARNING)
        warning_file_handler.setFormatter(formatter)
        root_logger.addHandler(warning_file_handler)

    # Set silent loggers to WARNING level
    if silent_names is not None:
        for logger_name in silent_names:
            logger = logging.getLogger(logger_name)
            logger.setLevel(logging.WARNING)



logger_name = "SamNotebook"
warning_file = HOME / f"data/logs/{logger_name}__warnings.log"
print(f"Saving warning logs to '.../{warning_file.relative_to(HOME)}'")

setup_logging(
    console_level=logging.INFO,
    warning_filename=warning_file.as_posix(),
    log_format="[ %(levelname)s ] %(name)s | %(funcName)s | %(message)s",
    silent_names=["matplotlib", "PIL"]
)

logger = logging.getLogger(logger_name)

Saving warning logs to '.../data/logs/SamNotebook__warnings.log'


## 🧪 Experiment data

* ``SPLIT_NAME``: Subset of samples to use in the model input prompts. Let empty for no filtering subset.

* `N_CLASS_SUBSAMPLES`: number of samples to select per class from annotations. Set to 0 to omit this parameter.

* `N_CLASS_RATIO`: proportion of samples to select per class from annotations. Must be in range `(0.0, 1.0]`. A value of 1 means the use of all samples. Set to 0 to omit this parameter.

* Annotation filenames:

    * `annotations_merged_rectified_split.csv`
    * `annotations_merged.parquet`

In [ ]:
#@title ✔️ Variable inputs


# Annotations subfolder
ANNOTATIONS_NAME = "Rectified_Polygons" # @param ["points","polygons", "Rectified_Polygons"]

# Annotations filename to load
ANNOTATIONS_FILENAME = "annotations_merged.parquet" # @param {"type":"string"}

# Annotations subset to look for in the "Split Name" column
SPLIT_NAME = "" # @param {"type":"string"}


# Subsample parameters
# -------------------------
# Number of per-class points to select
N_CLASS_SUBSAMPLES = 30 # @param {"type":"integer"}

# Ratio of per-class points to select
N_CLASS_RATIO = 0 # @param {"type":"number"}


# Params check
if N_CLASS_SUBSAMPLES <= 0:
    N_CLASS_SUBSAMPLES = None
if N_CLASS_RATIO <= 0:
    N_CLASS_RATIO = None
if len(SPLIT_NAME.strip()) == 0:
    SPLIT_NAME = None
if N_CLASS_SUBSAMPLES and N_CLASS_RATIO:
    raise ValueError("One of `N_CLASS_SUBSAMPLES` or `N_CLASS_RATIO` must be omited")

# Display selections
print("Annotations name:", ANNOTATIONS_NAME)
print("Annotations filename:", ANNOTATIONS_FILENAME)
print()
print("Split Name:", SPLIT_NAME)
print("Class subsamples:", N_CLASS_SUBSAMPLES)
print("Class ratio:", N_CLASS_RATIO)

Annotations name: Rectified_Polygons
Annotations filename: annotations_merged.parquet

Split Name: None
Class subsamples: 30
Class ratio: None


In [ ]:
#@title ✔️ Constants


# Directory Paths Config
# ==================================================================================
# Annotations data
ANNOTATIONS_PATH = HOME / f"data/annotations/{ANNOTATIONS_NAME}"

# Images files
IMAGES_PATH = ANNOTATIONS_PATH / "images/rectified"

# Output segmentation results
SEGMENTED_PATH = ANNOTATIONS_PATH / "images/segmented"
SEGMENTED_PATH.mkdir(parents=True, exist_ok=True)


# General Config
# ==================================================================================
# Mask with valid rectified pixels
VALID_MASK_FILE = HOME / "data/masks/rectify_valid_pixels_mask.png"

# Sorted class names
CLASSES_MAPPING = {
    "points": [ "Agua", "Humeda", "Roca", "Rompiente", "Seca" ],
    "polygons": [ "Agua", "Arena humeda", "Roca", "Rompiente", "Arena seca" ],
    "Rectified_Polygons": [ "Agua", "Arena humeda", "Roca", "Rompiente", "Arena seca" ],
}
CLASS_NAMES = CLASSES_MAPPING[ANNOTATIONS_NAME]

# Class colors
BACKGROUND_COLOR = [0, 0, 0]
BACKGROUND_LABEL = 255
PALETTE = [
    [1, 1, 255],
    [255, 255, 1],
    [255, 1, 255],
    [1, 255, 255],
    [255, 1, 1]
]


# Validations
# ==================================================================================
if len(CLASS_NAMES) != len(PALETTE):
    raise ValueError(
        f"Number of class names ({len(CLASS_NAMES)}) must match number of "
        f"palette colors ({len(PALETTE)})"
    )


# ========================================================================
# CONIFG INITIALIZATION
# ========================================================================
# Segmentation mapping
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}
IDX_TO_COLOR = {idx: color for idx, color in enumerate(PALETTE)}

# Add background color to the mapping
IDX_TO_COLOR.update({ BACKGROUND_LABEL: BACKGROUND_COLOR })  # Background color



print(f"""Valid mask file: '.../{VALID_MASK_FILE.relative_to(HOME)}'

Directories:
    * Annotations: '.../{ANNOTATIONS_PATH.relative_to(HOME)}'
    *      Images: '.../{IMAGES_PATH.relative_to(HOME)}'
    *   Segmented: '.../{SEGMENTED_PATH.relative_to(HOME)}'

Segmentation config:
    *     Classes: {CLASS_NAMES}
    *  Background: label={BACKGROUND_LABEL}, color={BACKGROUND_COLOR}
    * class_to_id: {CLASS_TO_IDX}
    * id_to_color: {IDX_TO_COLOR}
""")

Valid mask file: '.../data/masks/rectify_valid_pixels_mask.png'

Directories:
    * Annotations: '.../data/annotations/Rectified_Polygons'
    *      Images: '.../data/annotations/Rectified_Polygons/images/rectified'
    *   Segmented: '.../data/annotations/Rectified_Polygons/images/segmented'

Segmentation config:
    *     Classes: ['Agua', 'Arena humeda', 'Roca', 'Rompiente', 'Arena seca']
    *  Background: label=255, color=[0, 0, 0]
    * class_to_id: {'Agua': 0, 'Arena humeda': 1, 'Roca': 2, 'Rompiente': 3, 'Arena seca': 4}
    * id_to_color: {0: [1, 1, 255], 1: [255, 255, 1], 2: [255, 1, 255], 3: [1, 255, 255], 4: [255, 1, 1], 255: [0, 0, 0]}



# ✨ Definitions

In [ ]:
#@title ✔️ Utils
#   --> format_time                 [ OK ]
#   --> optimize_dataframe_dtypes   [ OK ]
#   --> save_prompts_figure         [ OK ]
#   --> convert_label2rgb           [ OK ]


def format_time(seconds):
    """Converts seconds into HH:MM:SS format."""
    # Calculate hours, minutes, and remaining seconds
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)

    # Format the string as HH:MM:SS (<total seconds>)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d} ({seconds:.4f} seconds)"


def optimize_dataframe_dtypes(
        df: pandas.DataFrame,
        dtype_hints: Optional[dict] = None
) -> pandas.DataFrame:
    """
    Optimize the data types of a pandas DataFrame based on provided hints and inferred types.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to optimize.

    dtype_hints : dict, optional
        A dictionary where keys are column names and values are the desired data types.
        Possible values are:

        - "category": Convert to categorical type.
        - "datetime": Convert to datetime type.
        - "integer", "signed", "unsigned", "float": Convert to numeric types with down casting.

        If None, no specific hints are applied and the function will infer types automatically.

    Returns
    -------
    pandas.DataFrame
        The DataFrame with optimized data types.
    """
    logger.debug("[Start] optimize_dataframe_dtypes")

    if dtype_hints is None:
        dtype_hints = {}

    # Get the initial memory usage in MB
    initial_memory = df.memory_usage(deep=True).sum() / (1024 ** 2)

    # Convert data types based on provided hints
    for col_name, col_hint in dtype_hints.items():
        if col_name not in df.columns:
            logger.warning(f"Column '{col_name}' not found in read dataset. Skipping dtype conversion.")
        else:
            try:
                # Get the current data type
                old_dtype = df[col_name].dtype

                # Attempt to change the data type
                if col_hint == "category":
                    df[col_name] = df[col_name].astype("category")
                elif col_hint == "datetime":
                    df[col_name] = pandas.to_datetime(df[col_name], errors="raise")
                elif col_hint in ("integer", "signed", "unsigned", "float"):
                    df[col_name] = pandas.to_numeric(df[col_name], downcast=col_hint, errors="raise")
                else:
                    logger.warning(f"Unsupported hint type '{col_hint}' for column '{col_name}'. Skipping.")

                # Log dtype change if different
                new_dtype = df[col_name].dtype
                if old_dtype != new_dtype:
                    logger.debug(f"Column '{col_name}' type changed from '{old_dtype}' to '{new_dtype}'")
            except ValueError as e:
                raise ValueError(f"Failed to convert column '{col_name}' with '{col_hint}' hint: {e}") from e

    # Data type to be ignored in the inference process
    ignore_dtypes = {"categorical", "datetime64", "datetime", "boolean", "bytes"}

    # Try to optimize the remaining columns
    remain_cols = set(df.columns) - set(dtype_hints.keys())
    for col_name in remain_cols:
        try:
            # Get the current data type
            old_dtype = df[col_name].dtype

            # Infer the data type
            type_label = pandas.api.types.infer_dtype(df[col_name], skipna=False)

            # Attempt to convert the column based on the inferred type
            if type_label == "integer":
                if (df[col_name] >= 0).all():
                    df[col_name] = pandas.to_numeric(df[col_name], downcast='unsigned')
                else:
                    df[col_name] = pandas.to_numeric(df[col_name], downcast='integer')
            elif type_label == "floating":
                df[col_name] = pandas.to_numeric(df[col_name], downcast="float")
            elif type_label == "string":
                try:
                    # Attempt datetime conversion
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", UserWarning)
                        df[col_name] = pandas.to_datetime(df[col_name])
                except Exception:
                    # If not a date, check if it's a good candidate for category
                    if df[col_name].nunique() / len(df) < 0.5:
                        df[col_name] = df[col_name].astype("category")
            else:
                if type_label not in ignore_dtypes:
                    logger.warning(f"Unsupported inferred type '{type_label}' for column '{col_name}'. Skipping.")

            # Log dtype change if modified
            new_dtype = df[col_name].dtype
            if old_dtype != new_dtype:
                logger.debug(f"Column '{col_name}' type changed, by inference, from '{old_dtype}' to '{new_dtype}'")
        except ValueError as e:
            raise ValueError(f"Failed to convert column '{col_name}' automatically: {e}") from e

    # Get the final memory usage in MB
    final_memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    memory_diff = initial_memory - final_memory
    memory_ratio = 100 * memory_diff / initial_memory
    logger.info(
        f"Memory usage (MB) -> Before={initial_memory:.3f}; After={final_memory:.3f}; "
        f"Saved={memory_diff:.3f} ({memory_ratio:.1f}%)"
    )

    logger.debug("[Finish] optimize_dataframe_dtypes")
    return df


def save_prompts_figure(
        image: numpy.ndarray,
        all_points: Dict[str, numpy.ndarray],
        selected_points: Dict[str, numpy.ndarray],
        output_file: Optional[Path] = None,
        split_name: Optional[str] = None,
        selection_params: Optional[dict] = None
) -> Optional[plt.Figure]:
    # Format the parameters used to select points
    params_str = ""
    if selection_params:
        params_str = "(" + ", ".join(f"{k}={v}" for k, v in selection_params.items()) + ")"

    # --- Input Prompts Figure ---
    fig, axs = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
    ax1, ax2 = axs

    # Axis with all annotated points
    ax1.imshow(image, cmap="gray")
    ax1.set_title(f"All points")
    for class_name, class_points in all_points.items():
        if class_points.size > 0:
            label = f"{class_name} ({len(class_points)})"
            ax1.scatter(class_points[:, 0], class_points[:, 1], label=label, s=2)
    ax1.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.05),
        ncol=len(all_points)//2,
        markerscale=2
    )

    # Axis with the selected points to make inference
    ax2.imshow(image, cmap="gray")
    ax2.set_title(f"Selected points {params_str}")
    for class_name, class_points in selected_points.items():
        if class_points.size > 0:
            label = f"{class_name} ({len(class_points)})"
            ax2.scatter(class_points[:, 0], class_points[:, 1], label=label, s=2)
    ax2.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.05),
        ncol=len(selected_points)//2,
        markerscale=2
        )

    # Figure adjust
    fig_suptitle = f"{image_filename}"
    if split_name:
        fig_suptitle += f" (split={split_name})"
    fig.suptitle(fig_suptitle, fontsize=12, fontweight="bold")
    fig.tight_layout()

    # Save the figure
    if output_file:
        output_file.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_file, dpi=300, bbox_inches="tight")
        plt.close(fig)
    else:
        return fig


def convert_label2rgb(
        label_array: numpy.ndarray,
        label_to_color: Dict[int, Sequence[int]]
) -> numpy.ndarray:
    """
    Converts a 2D label array of shape (height, width) with values in range [0, n_labels-1] into an RGB color image
    of shape (height, width) with values in range [0, 255].

    Parameters
    ----------
    label_array : numpy.ndarray
        2D array of integer labels.

    label_to_color : Dict[int, Sequence[int]]
        Dictionary mapping integer label values to RGB color sequences (e.g., {0: [255, 0, 0], 1: [0, 255, 0], ...}).

    Returns
    -------
    numpy.ndarray
        3D array representing the RGB color image, dtype=uint8.
    """
    logger.debug("[Start] convert_label2rgb")

    # Unique label array values
    uq_label_values = set(numpy.unique(label_array))

    # Check all array values are in the 'label_to_color' mapping
    missing_labels = uq_label_values - set(label_to_color.keys())
    if len(missing_labels) > 0:
        raise ValueError(f"Missing `label_array` values in the `label_to_color` mapping: {missing_labels}")

    # Create an empty RGB image
    height, width = label_array.shape
    color_image = numpy.empty((height, width, 3), dtype=numpy.uint8)

    # Assign colors to the RGB image based on the label values
    for label_value in uq_label_values:
        color = label_to_color[label_value]
        color_image[label_array == label_value] = color

    logger.debug("[Finish] convert_label2rgb")
    return color_image


In [ ]:
#@title ✔️ Processors
#   --> initialize_sam              [ OK ]
#   --> load_annotations_data       [ OK ]
#   --> subsample_points_by_class   [ OK ]


def initialize_sam(model_type: str, checkpoint_path: Path, device: str) -> SamPredictor:
    """Loads the SAM model and initializes the predictor."""
    logger.info(f"Loading SAM model ({model_type})")
    try:
        sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        sam.to(device=device)
        predictor = SamPredictor(sam)
        return predictor

    except FileNotFoundError:
        raise ValueError(f"Error: SAM Checkpoint not found at {checkpoint_path}")
    except Exception as e:
        raise ValueError(f"Error loading SAM model: {e}")


def load_annotations_data(data_file: Path) -> pandas.DataFrame:
    if not data_file.is_file():
        raise ValueError(f"File not found: '{data_file}'")

    file_extension = data_file.suffix.lower()

    # Parquet file
    if file_extension == ".parquet":
        logger.info(f"Data loaded as 'Parquet' file")
        df = pandas.read_parquet(data_file)

        # Rename Cx -> Cx Rectify, Cy -> Cy Rectify
        df = df.rename(columns={"Cx": "Cx Rectify", "Cy": "Cy Rectify"})

    # CSV file
    elif file_extension == ".csv":
        logger.info(f"Data loaded as 'CSV' file")

        # Data types for RAM optimization
        optimized_dtypes = {
            "Filename": "category",
            "Zona": "category",
            "Shape Name": "category",
            "Cx Rectify": "float",
            "Cy Rectify": "float",
            "Zona Id": "category",
            "Split Name": "category"
        }

        # Read only needed columns with specific data types
        df = pandas.read_csv(
            data_file,
            usecols=lambda col: col in optimized_dtypes.keys(),
            dtype=optimized_dtypes
        )

        # Columns validation
        required_cols = set(optimized_dtypes.keys())
        if not required_cols.issubset(df.columns):
            raise ValueError(
                f"Missing CSV columns. Expected: {required_cols}. Got: {df.columns}"
            )

    # Unsupported type
    else:
        raise ValueError(
            f"Unsupported '{file_extension.upper()}' extension file: '{data_file}'"
        )

    return optimize_dataframe_dtypes(df)


def subsample_points_by_class(
        image: numpy.ndarray,
        image_subset: pandas.DataFrame,
        image_filename: str,
        xy_columns: list,
        sorted_classes: List[str],
        select_ratio: Optional[float] = None,
        select_samples: Optional[int] = None,
        figure_file: Optional[Path] = None,
        split_name: Optional[str] = None
) -> Dict[str, Any]:
    """
    Subsamples points from the given image subset based on the specified ratio or number of samples.

    Parameters
    ----------
    image : numpy.ndarray
        The input image as a NumPy array.

    image_subset : pandas.DataFrame
        A DataFrame containing the subset of points for the given image.

    image_filename : str
        The filename of the image being processed.

    xy_columns : list
        List of column names representing the x and y coordinates of the points.
        Tipically, this would be ["Cx Rectify", "Cy Rectify"] or ["Cx", "Cy"].

    sorted_classes : List[str]
        A list of class names sorted in the desired order. Example: ["class1", "class2", "class3"].

    select_ratio : float, optional
        The ratio of points to select from each class. Must be in the range [0.1, 1.0].
        If provided, `select_samples` must be None.

    select_samples : int, optional
        The number of points to select from each class. If provided, `select_ratio` must be None.

    figure_file : Path, optional
        Path to save the figure showing all points and the selected points.
        If None (default), the figure is not saved in disk, but it is included in
        the returned dictionary as 'prompts_figure'.

    split_name : str, optional
        The name of the data split (e.g., "train", "val", "test"). This is used exclusively for
        the figure title and does not affect the subsampling process. Figure title format is as:
        "{image_filename} (split={split_name})".

    Returns
    -------
    Dict[str, Any]
        A dictionary containing:
        - "input_points": A NumPy array of the selected points for inference.
        - "target_labels": A NumPy array of the corresponding class labels for the selected points.
        - "selected_indices": A pandas Index of the selected point indices from the original DataFrame.
        - "prompts_figure": A matplotlib Figure object showing all points and the selected points.
    """
    # Check required columns
    required_cols = {"Shape Name", "Zona"}
    required_cols.update(xy_columns)
    if not required_cols.issubset(image_subset.columns):
        missing_cols = required_cols - set(image_subset.columns)
        raise ValueError(f"Missing required columns in dataframe: {missing_cols}")

    # Check the selection parameters
    if select_ratio and select_samples:
        raise ValueError("Only one of `select_ratio` or `select_samples` must be given")
    elif select_ratio and not (0.1 <= select_ratio <= 1.0):
            raise ValueError(f"Select ratio must be in range (0.0, 1.0]. Got: {select_ratio}")

    # Create a copy
    subset_df = image_subset.loc[:, list(required_cols)].copy()
    if subset_df.empty:
        raise ValueError(f"Empty image subset for '{image_filename}' image")

    # Check that annotated shapes are consistently points or polygon points
    uq_shape_names = subset_df["Shape Name"].unique()
    is_point_type_present = all(s == "point" for s in uq_shape_names)
    is_poly_type_present = all(s.startswith("poly_point_") for s in uq_shape_names)
    if not (is_point_type_present or is_poly_type_present):
        raise ValueError(
            f"All shapes must be either 'point' or 'poly_point'. Mixed shapes are not allowed."
        )

    # Setup the current shape mode
    if is_point_type_present:
        current_shape_mode = "point"
    else:
        current_shape_mode = "polygon"


    # ---- Main processing ----
    # Mapping for class names and points array
    all_points: Dict[str, numpy.ndarray] = {}
    selected_points: Dict[str, numpy.ndarray] = {}
    selected_indices: pandas.Index = pandas.Index([], dtype=image_subset.index.dtype)

    # Per-class processing
    for class_name, class_group in subset_df.groupby(by="Zona", observed=True):
        # Drop rows with NaN rectified coordinates
        class_name = str(class_name)
        class_df = class_group.dropna(subset=xy_columns)
        if class_df.empty:
            logger.warning(f"Empty '{class_name}' class points after dropping NaNs. Skipping")
            # selected_points[class_name] = numpy.empty((0, 2))
            continue

        # Subsample class points, if given any select param
        if select_ratio or select_samples:
            # Number of class samples to select
            if select_ratio:
                n_samples = int(len(class_df) * select_ratio)
            else:
                n_samples = min(select_samples, len(class_df))
                if select_samples > len(class_df):
                    logger.warning(
                        f"The samples to select ({select_samples}) is greater than "
                        f"the total '{class_name}' class points ({len(class_df)}). "
                        f"Selecting all points"
                    )
        # No subsampling
        else:
            n_samples = len(class_df)


        # Subsample random points
        if len(class_df) != n_samples:
            # Random point indices
            random_indices = numpy.random.choice(
                a=class_df.index,
                size=n_samples,
                replace=False   # ensures no duplicate points
            )
            point_indices = pandas.Index(random_indices)

            # Display subsampling
            subsample_ratio = ( n_samples / len(class_df) ) * 100
            logger.info(
                f"Class '{class_name}' points were reduced by {subsample_ratio:.3f}% "
                f"from {len(class_df)} to {n_samples} samples"
            )
        else:
            point_indices = class_df.index

        # Select random points.
        # Type: numpy.ndarray of shape (n_samples, 2) with the X and Y coordinates
        selected_class_points = class_df.loc[point_indices, xy_columns].to_numpy()

        # Save the class points
        all_points[class_name] = class_df.loc[:, xy_columns].to_numpy()
        selected_points[class_name] = selected_class_points
        selected_indices = selected_indices.union(point_indices, sort=True)

    # Compute comparative prompts figure
    # import pdb; pdb.set_trace()
    fig: Optional[plt.Figure] = None
    if not (select_ratio or select_samples):
        logger.warning("Skipping prompts figure creation as no points were subsampled")
    else:
        if select_ratio:
            params = {"ratio": select_ratio}
        else:
            params = {"class_subsamples": select_samples}

        fig = save_prompts_figure(
            image=image,
            all_points=all_points,
            selected_points=selected_points,
            output_file=figure_file,
            split_name=split_name,
            selection_params=params
        )

    # Convert selected points to match the expected model input
    all_selected_points: List[numpy.ndarray] = []
    all_selected_labels: List[numpy.ndarray] = []
    for class_name, class_points in selected_points.items():
        if class_name not in sorted_classes:
            raise ValueError(
                f"Class '{class_name}' is not presented in the given "
                f"sorted class names: {sorted_classes}"
            )
        class_id = sorted_classes.index(class_name)
        labels = numpy.full(shape=len(class_points), fill_value=class_id)
        all_selected_points.append(class_points)
        all_selected_labels.append(labels)

    # Final output arrays where "n_samples" is equivalent to: n_classes * n_selected_class_points
    input_points = numpy.vstack(all_selected_points)        # Shape: (n_samples, 2)
    target_labels = numpy.concatenate(all_selected_labels)  # Shape: (n_samples,)

    return {
        "input_points": input_points,
        "target_labels": target_labels,
        "selected_indices": selected_indices,
        "prompts_figure": fig
    }


In [ ]:
#@title ✔️ SAM predictor


class SparsePointsSamPredictor:

    def __init__(
            self,
            predictor: Any,
            max_input_points: Optional[int] = None, # 11000
            device: Literal["cpu", "cuda"] = "cpu"
    ):
        # Check the input processing device
        device = device.lower().strip()
        if device not in ["cpu", "cuda"]:
            raise ValueError(f"Invalid device. Possible values are: 'cpu' or 'cuda'. Got: '{device}'")
        if device == "cuda" and not torch.cuda.is_available():
            logger.warning("CUDA is not available on this system. Using CPU instead")
            device = "cpu"

        # Public attributes
        # ---------------------------------------------------
        self.predictor = predictor
        self.max_input_points = max_input_points            # Maximum SAM input prompts (limited for vRAM issues)
        self.device = device

        self.processing_times: Dict[str, float] = {}        # General workflow processing times in seconds
        self.prediction_times: Dict[str, float] = {}        # Per-class prediction times in seconds
        self.mask_logits: Optional[torch.Tensor] = None     # Mask logits predicted. Shape: (n_classes, H, W)
        self.class_to_points: Dict[str, numpy.ndarray] = {} # Mapping from class names to (x, y) coordinates

        # Private attributes
        # ---------------------------------------------------
        self._class_names: List[str] = []                       # Class names for the points
        self._image: Optional[numpy.ndarray] = None             # Image to predict on

    def __str__(self):
        return (
            f"{self.__class__.__name__}("
            f"predictor={self.predictor.__class__.__name__}, "
            f"max_points={self.max_input_points}, "
            f"device={self.device}"
            ")"
        )

    def __repr__(self):
        return str(self)


    # Public methods
    # --------------------------------------------------------------
    def set_input(
            self,
            image: numpy.ndarray,
            input_points: numpy.ndarray,
            target_labels: numpy.ndarray,
            sorted_classes: List[str],
            image_format: Literal["RGB", "BGR"] = "RGB"
    ):
        """
        Set the input data for the predictor.

        Parameters
        ----------
        image : numpy.ndarray
            The image to predict on. It should be a 3D uint8 array with shape (height, width, channels).

        input_points : numpy.ndarray
            2D array of input points with shape (N, 2) where N is the number of points.
            Each point should be represented as (x, y) pixel coordinates in the image.

        target_labels : numpy.ndarray
            1D array of class labels corresponding to the input points. Each label should be an
            integer representing the class index in `sorted_class_names`.

        sorted_classes : List[str]
            List of class names sorted in the order of their indices. The length of this list should
            match the number of unique labels in `target_labels`.

        image_format : str, optional
            The color format of the image channels. It can be either "RGB" or "BGR".
            Default is "RGB".
        """
        # input_points = numpy.column_stack((input_x_points, input_y_points))  # Shape (N, 2) as (x, y) coordinates

        # Check the image format
        image_format = image_format.upper().strip()
        if image_format not in ["RGB", "BGR"]:
            raise ValueError(f"Invalid image format. Possible values are: 'RGB' or 'BGR'. Got: '{image_format}'")

        # Check the shape of input points
        if input_points.ndim != 2 or input_points.shape[1] != 2:
            raise ValueError(
                f"Invalid input points shape. Expected 2D array with shape (N, 2) "
                f"as (x, y) coordinates. Got: {input_points.shape}"
            )
        if len(input_points) == 0:
            raise ValueError("Input points arrays must not be empty")

        # Check the shape of target labels
        if target_labels.ndim != 1:
            raise ValueError(
                f"Invalid target labels shape. Expected 1D array with shape (N,) "
                f"as class indices. Got: {target_labels.shape}"
            )

        # Check that input points and target labels match
        if input_points.shape[0] != target_labels.shape[0]:
            raise ValueError(
                f"Input points and target labels must have the same number of elements. "
                f"Got: input_points={input_points.shape[0]}, target_labels={target_labels.shape[0]}"
            )

        # Check unique target labels with class names
        uq_target_labels = set(numpy.unique(target_labels).tolist())
        n_classes = len(sorted_classes)
        if len(uq_target_labels) != n_classes:
            logger.warning(
                f"Number of unique target labels ({len(uq_target_labels)}) does not match "
                f"the number of sorted class names ({n_classes})"
            )

        # Check the target labels are within the range of class name indices
        # if uq_target_labels != set(range(n_classes)):
        diff = uq_target_labels - set(range(n_classes))
        if len(diff) > 0:
            raise ValueError(
                f"Target labels must be in the range of class names indices. "
                f"Got: unique_labels={uq_target_labels}. Expected range: [0, {n_classes - 1}]"
            )

        # Check for invalid x and y coordinates
        height, width = image.shape[:2]
        x_pts, y_pts = input_points[:, 0], input_points[:, 1]
        invalid_x_mask = (x_pts < 0) | (x_pts >= width)     # A valid x is in the range [0, width - 1]
        invalid_y_mask = (y_pts < 0) | (y_pts >= height)    # A valid y is in the range [0, height - 1]
        invalid_points_mask = invalid_x_mask | invalid_y_mask
        if numpy.any(invalid_points_mask):
            invalid_points = input_points[invalid_points_mask].tolist()
            raise ValueError(
                f"Input points must be valid pixel coordinates within the image dimensions. "
                f"Image shape: {image.shape}. Invalid points found: {invalid_points}. "
                f"Valid x range: [0, {width - 1}], valid y range: [0, {height - 1}]"
            )

        # Preprocess the input data
        self._preprocess(image, input_points, target_labels, sorted_classes, image_format)

    def make_inference(self, use_background_prompts: bool = False):
        """
        Runs the SAM predictor on the input image and returns the predicted logits and labels.

        Parameters
        ----------
        use_background_prompts : bool, optional
            If True, include background points (negative points) in the input prompts.
            Default is False.

        Returns
        -------
        Dict[str, numpy.ndarray]
            Dictionary containing the predictions:

            *   "logits" (numpy.ndarray): predicted raw logits for each class with
                shape (n_classes, height, width).

            *   "labels" (numpy.ndarray): post-processed logits for each pixel with shape (height, width).
                This is obtained by taking the argmax over the raw logits along the class dimension.
        """
        # Check if the image embedding has been set
        if self._image is None:
            raise ValueError("Image embedding has not been set. Use 'set_input' method first.")

        # Run predictions for each class
        if use_background_prompts:
            logger.info("Running per-class predictions using background input prompts")
        else:
            logger.info("Running per-class predictions")
        self._predict(use_background_prompts)

        # Post-process the predictions
        logger.info("Post-processing the predictions")
        results = self._postprocess()

        # Clear GPU memory
        if self.device == "cuda":
            torch.cuda.empty_cache()

        return results


    # Private methods
    # --------------------------------------------------------------
    def _preprocess(
            self,
            image: numpy.ndarray,
            input_points: numpy.ndarray,
            target_labels: numpy.ndarray,
            sorted_classes: List[str],
            image_format: str
    ):
        # Update input attributes
        self._class_names = sorted_classes
        self.class_to_points = {}
        self._image = None

        # Compute mapping from class names to coordinates
        # TODO: What happens if there are no input points for a class?
        for idx, class_name in enumerate(sorted_classes):
            # Check if there are input points for the class
            if idx not in target_labels:
                logger.warning(f"No input points for class '{class_name}' (id={idx})")
                continue

            class_coords = input_points[target_labels == idx]
            self.class_to_points[class_name] = class_coords
            logger.debug(f"Total points for class '{class_name}': {len(class_coords)}")
        logger.info(f"Total input points: {len(input_points)}")

        # Check if the input points exceed the maximum allowed for SAM
        if self.max_input_points and (len(input_points) > self.max_input_points):
            logger.warning(
                f"Total input points ({len(input_points)}) exceed the maximum "
                f"value configured for SAM ({self.max_input_points}). "
                f"Possible {self.device.upper()} out of memory issues."
            )

        # Set SAM embedding
        start_time = time.time()
        try:
            logger.info("Calculating the image embeddings")
            self.predictor.set_image(image=image, image_format=image_format)
            self._image = image
        except Exception as e:
            if self.device == "cuda":
                torch.cuda.empty_cache()
            raise ValueError(f"Fail to calculate image embedding: {e}.") from e
        embeddings_time = time.time() - start_time
        self.processing_times["embeddings"] = embeddings_time
        logger.info(f"Image embeddings calculated in {embeddings_time:.3f} seconds")

        # Initialize a logit tensor for all classes
        # Fill with a very low value so that unpredicted classes don't win argmax
        height, width, _ = self._image.shape
        self.mask_logits = torch.full(
            size=(len(self._class_names), height, width),
            fill_value=-float("inf"),
            device=self.device,
            dtype=torch.float32
        )

    def _predict(self, use_background_prompts: bool = False):
        """
        Runs SAM predictor for each class present in the image's training dataframe subset.

        Returns
        -------
        tuple
            Two elements tuple:

            1. Dictionary containing the below keys. This might be None if any failure occur:

            - "proba": tensor with class probabilities of shape (n_classes, height, width).
            - "labels": tensor with argmax predictions of shape (height, width).
            - "class_points": dictionary mapping class names to their corresponding selected
            coordinates as `numpy` arrays of shape (N, 2) with (x, y) coordinates.

            2. Boolean flag to know if the process was successful.
        """
        # The accumulated prediction time for all classes
        final_prediction_time = 0.0

        # Per-class inference
        for positive_class in self._class_names:
            if positive_class not in self.class_to_points:
                logger.warning(f"No input points for class '{positive_class}'. Skipping predictions.")
                continue

            # Get positive class data
            class_idx = self._class_names.index(positive_class)
            positive_points = self.class_to_points[positive_class]
            logger.debug(f"Predicting '{positive_class}' class (id={class_idx})")

            # Create the input points and labels
            input_points_list = []
            input_labels_list = []
            input_points_list.append(positive_points)
            input_labels_list.append(numpy.ones(shape=len(positive_points), dtype=int))

            # Include negative points in the input points prompt
            if use_background_prompts:
                negative_classes = set(self._class_names) - {positive_class}
                negative_list = [
                    self.class_to_points[negative_class]
                    for negative_class in negative_classes
                    if negative_class in self.class_to_points
                ]
                negative_points = numpy.vstack(negative_list)
                if len(negative_points) > 0:
                    input_points_list.append(negative_points)
                    input_labels_list.append(numpy.zeros(shape=len(negative_points), dtype=int))
                    logger.debug(f"Total background points for class '{positive_class}': {len(negative_points)}")
                else:
                    logger.warning(f"No background points found for class '{positive_class}'")

            # Create the inputs as arrays
            input_points = numpy.vstack(input_points_list)
            input_labels = numpy.concatenate(input_labels_list)

            logger.debug(f"Running SAM predictions with {len(input_points)} input prompts")
            try:
                with torch.no_grad():
                    #   masks_logits --> array of shape: (n_masks, height, width)
                    #         scores --> array of shape: (n_masks, )
                    # low_res_logits --> array of shape: (n_masks, 256, 256)
                    start_time = time.time()
                    masks_logits, scores, low_res_logits = self.predictor.predict(
                        point_coords=input_points,
                        point_labels=input_labels,
                        return_logits=True,         # Get un-thresholded masks logits instead of a binary mask
                        multimask_output=False,     # Output a single best mask/logit (n_masks=1)
                    )
                    predictions_time = time.time() - start_time
                    # self.processing_times["predictions"] = predictions_time
                    self.prediction_times[positive_class] = predictions_time
                    final_prediction_time += predictions_time
                    logger.debug(f"Class '{positive_class}' prediction time: {predictions_time:.3f} seconds")

                    # Check expected output shapes
                    expected_masks_shape = (1, self._image.shape[0], self._image.shape[1])
                    if masks_logits.shape != expected_masks_shape:
                        raise ValueError(
                            f"Unexpected predicted masks shape. Expected: {expected_masks_shape}. "
                            f"Predicted: {masks_logits.shape}"
                        )

                # Place the logits in the class index slot
                self.mask_logits[class_idx, :, :] = torch.from_numpy(masks_logits).squeeze()

            except Exception as e:
                raise ValueError(f"Fail to run SAM predictions for class '{positive_class}': {e}") from e

        # Update the final prediction time
        self.processing_times["predictions"] = final_prediction_time
        logger.info(f"Image predictions calculated in {final_prediction_time:.3f} seconds")

    def _postprocess(self) -> Dict[str, numpy.ndarray]:
        # Check if argmax resulted in -inf (i.e., no class positively predicted)
        logit_max_values, logit_max_indices = torch.max(input=self.mask_logits, dim=0)
        n_unpredicted = torch.sum(logit_max_values == -float("inf")).item()
        if n_unpredicted > 0:
            logger.warning(
                f"There are {n_unpredicted} pixels with no class predicted (max logit was -inf)."
                f"They will be assigned an arbitrary class index by argmax (usually 0)."
            )

        # Get labels from predicted logits
        logger.debug("Generating prediction map using argmax on aligned logits")
        mask_labels = torch.argmax(input=self.mask_logits, dim=0)   # Shape: (H, W)

        return {
            "logits": self.mask_logits.cpu().numpy(),
            "labels": mask_labels.cpu().numpy().astype(int)
        }


# ✨ Segmentation Flow

In [ ]:
#@title ✔️ SAM instance


logger.info(f"Using SAM checkpoint from '.../{WEIGHTS_FILE.relative_to(HOME)}'")
device = "cuda" if torch.cuda.is_available() else "cpu"
sam_predictor = initialize_sam(
    model_type="vit_h",
    checkpoint_path=WEIGHTS_FILE,
    device=device
)
model = SparsePointsSamPredictor(predictor=sam_predictor, max_input_points=11000, device=device)
print("\nCustom Model:", model)

[ INFO ] SamNotebook | <cell line: 0> | Using SAM checkpoint from '.../data/models/weights/sam_vit_h_4b8939.pth'
[ INFO ] SamNotebook | initialize_sam | Loading SAM model (vit_h)

Custom Model: SparsePointsSamPredictor(predictor=SamPredictor, max_points=11000, device=cuda)


In [ ]:
#@title ✔️ Load annotations


# Load dataframe
dataset_file = ANNOTATIONS_PATH / ANNOTATIONS_FILENAME
logger.info(f"Loading dataset from: '.../{dataset_file.relative_to(HOME)}'")
annotations_df = load_annotations_data(data_file=dataset_file)
total_samples = len(annotations_df)
logger.info(f"Total data samples: {total_samples}")

# Filter data by split value, if needed
if SPLIT_NAME is not None:
    # Select the corresponding split samples
    split_mask = annotations_df["Split Name"] == SPLIT_NAME
    split_df = annotations_df.loc[split_mask].copy()

    # ---> This is not recommended as the indices sequence is lost
    # split_df = annotations_df.set_index("Split Name").loc[SPLIT_NAME].reset_index()

    split_samples = len(split_df)
    split_ratio = (split_samples / total_samples) * 100
    logger.info(
        f"Using '{SPLIT_NAME.upper()}' split with {split_samples} ({split_ratio:.1f}%) samples"
    )
else:
    split_samples = total_samples
    split_df = annotations_df.copy()
    logger.info("Using ALL samples")


print("\n----->  Split dataFrame info  <-----")
split_df.info(memory_usage="deep")
print()

[ INFO ] SamNotebook | <cell line: 0> | Loading dataset from: '.../data/annotations/Rectified_Polygons/annotations_merged.parquet'
[ INFO ] SamNotebook | load_annotations_data | Data loaded as 'Parquet' file
[ INFO ] SamNotebook | optimize_dataframe_dtypes | Memory usage (MB) -> Before=12.440; After=12.440; Saved=0.000 (0.0%)
[ INFO ] SamNotebook | <cell line: 0> | Total data samples: 1629761
[ INFO ] SamNotebook | <cell line: 0> | Using ALL samples

----->  Split dataFrame info  <-----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1629761 entries, 0 to 1629760
Data columns (total 6 columns):
 #   Column        Non-Null Count    Dtype   
---  ------        --------------    -----   
 0   Filename      1629761 non-null  category
 1   Capture Name  1629761 non-null  category
 2   Shape Name    1629761 non-null  category
 3   Cx Rectify    1629761 non-null  uint16  
 4   Cy Rectify    1629761 non-null  uint16  
 5   Zona          1629761 non-null  category
dtypes: category(4), uint16(

In [ ]:
#@title ✔️ Preprocessing


# Load valid segmentation mask
valid_mask = io.imread(VALID_MASK_FILE, as_gray=True) > 0

# (x, y) coordinate columns
xy_columns = ["Cx Rectify", "Cy Rectify"]

# Directory to save the comparative of all vs selected points
figures_path = ANNOTATIONS_PATH / "images/sam_input_prompt_figures"

# Mapping of filename and the selected sample indicies
file_to_indices: Dict[str, pandas.Index] = {}

# Display setup
print(f"""Preprocessing setup
    * Valid mask shape: {valid_mask.shape}
    * (x, y) columns: {xy_columns}
    * Figures path: {figures_path.relative_to(HOME)}
""")

# Suffix to the results directory
name_format = "sam_{split}_{class_samples}"
split_str = SPLIT_NAME if SPLIT_NAME else "all"
model_name = name_format.format(
    split=SPLIT_NAME or "all",
    class_samples=N_CLASS_SUBSAMPLES or N_CLASS_RATIO or "all"
)
print(f"""\n
Model name results:
  * Format: sam_<split>_<class_samples>_<bg_prompts>
  * Name: {model_name}
""")

In [ ]:
#@title ✔️ Segment


# Create a custom handler to show if any warning was displayed
warning_occurred = False
final_warning = False
class WarningFlagHandler(logging.Handler):
    def emit(self, record):
        global warning_occurred
        if record.levelno >= logging.WARNING:
            warning_occurred = True
if not any(isinstance(h, WarningFlagHandler) for h in logger.handlers):
    logger.addHandler(WarningFlagHandler())
    logger.info("Custom 'WarningFlagHandler' handler added")
print()


# Set global seed and variables
# numpy.random.seed(23)
image_times: Dict[str, float] = {}
n_images = split_df["Filename"].nunique()
n_count = 0

# Per-image process
for image_filename, image_df in split_df.groupby(by="Filename", observed=True):
    n_count += 1
    image_filename = str(image_filename)
    print(f"< ==========  [START] {n_count}/{n_images} '{image_filename}'  ========== >")
    start_time = time.time()

    # Input Data
    # ================================================================================
    # Load image
    rectify_filename = image_filename.replace("undist.jpg", "rectified.tif")
    image_file = IMAGES_PATH / rectify_filename
    image = io.imread(image_file)

    # Display images info
    image_samples = len(image_df)
    image_ratio = (image_samples / split_samples) * 100
    logger.info(f"Image shape: {image.shape}")
    logger.info(f"Image samples (split={SPLIT_NAME}): {image_samples} ({image_ratio:.1f}%)")

    # Create input prompts figure file
    figure_file = figures_path / model_name / image_filename.replace(".tif", ".png")
    if not SPLIT_NAME:
        logger.info("Using all dataframe points.")

    # Select input points
    subsample_data = subsample_points_by_class(
        image=image,
        image_subset=image_df,
        image_filename=image_filename,
        xy_columns=xy_columns,
        sorted_classes=CLASS_NAMES,
        select_ratio=N_CLASS_RATIO,
        select_samples=N_CLASS_SUBSAMPLES,
        figure_file=figure_file,
        split_name=SPLIT_NAME,
    )

    # Store the selected indices
    selected_image_indices = subsample_data["selected_indices"].to_numpy()
    file_to_indices[rectify_filename] = selected_image_indices
    # selected_df_indices = selected_df_indices.union(selected_image_indices, sort=True)
    logger.info(f"Append {len(selected_image_indices)} image indices")

    # Model
    # ================================================================================
    # Set the inference input
    model.set_input(
        image=image,
        input_points=subsample_data["input_points"],
        target_labels=subsample_data["target_labels"],
        sorted_classes=CLASS_NAMES,
        image_format="RGB"
    )

    # Make inference
    # use_negative=True  --> Use both foreground and background as input prompts
    # use_negative=False --> Use only foreground as input prompts
    for use_negative in (True, False):
        # Predict
        sam_results = model.make_inference(use_background_prompts=use_negative)
        label_mask = sam_results["labels"]
        logit_mask = sam_results["logits"]

        # Output artifacts directory
        results_name = model_name
        if use_negative:
            results_name += "_negatives"
        output_path = SEGMENTED_PATH / results_name
        output_path.mkdir(parents=True, exist_ok=True)
        logger.info(f"SAM results saved at: '.../{output_path.relative_to(HOME)}'")

        # Mask out invalid rectified pixels
        label_mask[~valid_mask] = BACKGROUND_LABEL

        # Create and save a color mask
        color_mask_file = output_path / rectify_filename.replace(".tif", "_segmented.png")
        color_mask = convert_label2rgb(label_array=label_mask, label_to_color=IDX_TO_COLOR)
        io.imsave(fname=color_mask_file, arr=color_mask)

    # Display model processing times
    s = "\n".join(f"    * {k}: {v:.4f}" for k, v in model.processing_times.items())
    print(f"\nProcessing times (seconds):\n{s}")

    # Display model prediction times
    s = "\n".join(f"    * {k}: {v:.4f}" for k, v in model.prediction_times.items())
    print(f"\nPrediction times (seconds):\n{s}")

    # Health status
    if warning_occurred:
        icon = "⚠️"
        img_key = "⚠️ " + image_filename
        final_warning = True
        warning_occurred = False
    else:
        icon = "🟢"
        img_key = image_filename

    # Get the image execution time
    execution_time = time.time() - start_time
    image_times[img_key] = execution_time

    # Display final info
    print(f"\n{icon} Image execution time: {format_time(seconds=execution_time)}")
    print(f"< ==========  [FINISH] {n_count}/{n_images} '{image_filename}'  ========== >")
    print("\n\n")
    # break


# Save the selected dataframe indices
numpy.savez(
    file=SEGMENTED_PATH / f"indices__{model_name}.npz",
    **file_to_indices
)

# Finished info
if final_warning:
    print("⚠️ Processing finished with WARNINGS")
else:
    print("🟢 Processing finished successfully for all images")
s = "\n".join(f"    * {k}: {v:.4f}" for k, v in image_times.items())
print(f"\nImage times (seconds):\n{s}")

[ INFO ] SamNotebook | <cell line: 0> | Custom 'WarningFlagHandler' handler added

< ==========  [START] 1/20 '0108_0154_18_09_04_17_30_mean_rectified.tif'  ========== >
[ INFO ] SamNotebook | <cell line: 0> | Image shape: (480, 450, 3)
[ INFO ] SamNotebook | <cell line: 0> | Image samples (split=None): 83428 (5.1%)
[ INFO ] SamNotebook | <cell line: 0> | Using all dataframe points.
[ INFO ] SamNotebook | subsample_points_by_class | Class 'Agua' points were reduced by 0.044% from 67519 to 30 samples
[ INFO ] SamNotebook | subsample_points_by_class | Class 'Arena humeda' points were reduced by 0.574% from 5223 to 30 samples
[ INFO ] SamNotebook | subsample_points_by_class | Class 'Arena seca' points were reduced by 7.979% from 376 to 30 samples
[ INFO ] SamNotebook | subsample_points_by_class | Class 'Roca' points were reduced by 1.538% from 1951 to 30 samples
[ INFO ] SamNotebook | subsample_points_by_class | Class 'Rompiente' points were reduced by 0.359% from 8359 to 30 samples
[ INF